# Run EA-MD-QD preprocessing

This notebook runs the maintained `routine_data.py` script for the Euro Area aggregate.

Chosen settings:
- Country: `EA`
- Frequency: `QM` quarterly-aggregated panel
- Transformations: `light`
- Imputation method: `0` ragged-edge imputation only
- Number of factors: `99` automatic Bai-Ng choice
- Algorithm: `SW` Stock-Watson EM


In [ ]:
from pathlib import Path
import builtins
import importlib
import os
import sys

import pandas as pd


def find_ea_folder() -> Path:
    """Find the EA-MD-QD data folder whether Jupyter starts here or at project root."""
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "Data" / "EA-MD-QD-2026-04",
        cwd.parent / "Data" / "EA-MD-QD-2026-04",
    ]
    for candidate in candidates:
        if (candidate / "EAdata.xlsx").exists() and (candidate / "routine_data.py").exists():
            return candidate
    raise FileNotFoundError("Could not find EAdata.xlsx and routine_data.py.")


EA_DIR = find_ea_folder()
os.chdir(EA_DIR)
sys.path.insert(0, str(EA_DIR))
print(f"Working directory: {EA_DIR}")

# Import after changing directory because routine_data.py stores os.getcwd()
# in module-level variables used later for reading and saving files.
import routine_data
importlib.reload(routine_data)

answers = iter([
    "n",      # do not use all default countries
    "EA",     # Euro Area aggregate
    "QM",     # quarterly aggregated panel
    "light",  # light transformations
    "y",      # impute
    "0",      # ragged-edge imputation only
    "99",     # automatic number of factors
    "SW",     # Stock-Watson EM algorithm
])


def scripted_input(prompt: str = "") -> str:
    answer = next(answers)
    print(f"{prompt}{answer}")
    return answer


old_input = builtins.input
try:
    builtins.input = scripted_input
    Xc = routine_data.main()
finally:
    builtins.input = old_input

output_path = EA_DIR / "data_TR2" / "EAdataQM_TR2.xlsx"
print(f"\nOutput file: {output_path}")
print(f"Exists: {output_path.exists()}")


In [ ]:
# Quick output check
output_path = EA_DIR / "data_TR2" / "EAdataQM_TR2.xlsx"
ea_processed = pd.read_excel(output_path)

numeric = ea_processed.drop(columns=["Date"]).select_dtypes("number")
non_finite = (~pd.DataFrame(numeric).applymap(lambda x: pd.notna(x) and abs(x) != float("inf"))).sum().sum()

print(f"Shape: {ea_processed.shape}")
print(f"Date range: {ea_processed['Date'].min()} to {ea_processed['Date'].max()}")
print(f"Maximum missing share: {numeric.isna().mean().max():.2%}")
print(f"Non-finite cells, including +/-inf: {non_finite}")
ea_processed.head()
